# KI

## Datenaufbereitung

### Trainingsdaten laden

In [ ]:
# KI Daten einlesen 

import pandas as pd
import os

# Pfad zu den Daten
base_path = r'c:\Users\samso\Documents\00 Uni\Semester 9\sonar_mpa\data\messung_15_12_25'
files = [
    'grabel_training_data.csv',
    'sand_training_data.csv', 
    'stone_training_data.csv'
]

# Daten einlesen
data_frames = []
for file in files:
    file_path = os.path.join(base_path, file)
    df = pd.read_csv(file_path, sep=';')                # Einlesen der CSV-Dateien, Trennzeichen ist Semikolon
    data_frames.append(df)

# Zusammenfügen zu einem einzigen DataFrame
full_df = pd.concat(data_frames, ignore_index=True)

# Filtern der Spalten: Wir wollen 'class_name', 'Frequency' und alle 'S_...' Spalten
amplitude_cols = [col for col in full_df.columns if col.startswith('S_')]               # Identifizieren der Spalten, die mit 'S_' beginnen
selected_cols = ['class_name', 'Frequency'] + amplitude_cols                        # Auswahl der gewünschten Spalten
final_df = full_df[selected_cols]

# Anzeigen der ersten Zeilen und der Größe des DataFrames
print(f"Form des DataFrames: {final_df.shape}")
display(final_df.head())

### Ausgabe vo 20 Zeitreihen pro klasse in ein Diagramm

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Eindeutige Kombinationen aus Klasse und Frequenz finden
combinations = final_df[['class_name', 'Frequency']].drop_duplicates().values
# Sortieren für konsistente Reihenfolge
combinations = sorted(combinations, key=lambda x: (x[0], x[1]))

num_plots = len(combinations)

# Plot erstellen
fig, axes = plt.subplots(num_plots, 1, figsize=(15, 5*num_plots), sharex=True)

# Falls nur eine Kombination existiert, axes in eine Liste umwandeln
if num_plots == 1:
    axes = [axes]

for i, (class_name, freq) in enumerate(combinations):
    ax = axes[i]
    
    # Filterung für die aktuelle Klasse und Frequenz
    subset = final_df[(final_df['class_name'] == class_name) & (final_df['Frequency'] == freq)]
    
    # Zufällige Auswahl von 20 Proben (oder weniger, wenn nicht genug vorhanden sind)
    n_samples = min(20, len(subset))
    if n_samples > 0:
        # sample() nutzen, um zufällige Zeilen zu wählen
        sampled_data = subset.sample(n_samples, random_state=42)
        
        # Amplituden-Daten extrahieren (ohne class_name und Frequency)
        # Wir nutzen values um nur die reinen Zahlen zu plotten
        amplitudes = sampled_data.drop(columns=['class_name', 'Frequency']).values
        
        # Plotten jeder Zeitreihe
        for row in amplitudes:
            ax.plot(row, alpha=0.5, linewidth=1)
            
    ax.set_title(f'Zeitreihen für Klasse: {class_name}, Frequenz: {freq} ({n_samples} Proben)')
    ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Zeitindex')
plt.tight_layout()
plt.show()